# Retrieval-Augmented Generation

Companion notebook for the [RAG lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/04-retrieval-augmented-generation).

**The idea in one sentence.** RAG grounds an LLM in *your* data: **chunk** documents,
**retrieve** the chunks most relevant to the query, and **stuff** only those into the
prompt — so the model answers from retrieved facts instead of (possibly wrong) parametric
memory.

The pipeline:

- **Chunk** the source into overlapping passages.
- **Retrieve** the top-k chunks by similarity to the query.
- **Generate** an answer conditioned on those chunks.

We build a minimal RAG pipeline from scratch, **validate that retrieval surfaces the
relevant chunk and that recall@k behaves**, then cover the gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A minimal RAG pipeline

We reuse the toy embedder, add **chunking**, retrieve the top-k chunks for a query, assemble them into a context prompt, and run a mock `generate` that answers *only* from the retrieved context (extractive). No API key needed — the point is the pipeline shape.

In [ ]:
DOC = (
    'Our return policy allows refunds within 30 days of purchase. '
    'To request a refund, email support with your order number. '
    'Standard shipping takes 3 to 5 business days. '
    'International shipping can take up to 2 weeks. '
    'You can reset your password from the account settings page.'
)

def chunk(text, size=8, overlap=2):
    words = text.split()
    out, i = [], 0
    while i < len(words):
        out.append(' '.join(words[i:i + size]))
        i += size - overlap
    return out

chunks = chunk(DOC)
for c in chunks:
    print('-', c)

In [ ]:
def tokenize(s):
    return re.findall(r'[a-z]+', s.lower())

vocab = sorted({w for c in chunks for w in tokenize(c)})
index = {w: i for i, w in enumerate(vocab)}
def embed(text):
    v = np.zeros(len(vocab))
    for w in tokenize(text):
        if w in index:
            v[index[w]] += 1.0
    return v
def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(a @ b / (na * nb))

chunk_vecs = np.array([embed(c) for c in chunks])

def retrieve(query, k=2):
    q = embed(query)
    sims = [cosine(q, c) for c in chunk_vecs]
    return [chunks[i] for i in np.argsort(sims)[::-1][:k]]

def generate(query, k=2):
    context = retrieve(query, k)
    prompt = 'Answer using ONLY this context:\n' + '\n'.join(context) + f'\nQ: {query}\nA:'
    print(prompt)
    print('\n[mock answer would be grounded in the context above]')

generate('how long do refunds take to request')

### Validate: retrieval surfaces the relevant chunk

RAG only works if the retriever pulls the chunk that actually answers the query. We ask a
refund question and confirm the top-retrieved chunk mentions refunds, ranked by
similarity — the grounding the generator will condition on.

In [ ]:
hits = retrieve('how do I get a refund', k=2)   # returns a ranked list of chunk strings
print('query "how do I get a refund" -> top chunks:')
for c in hits:
    print(f'  {c}')
assert 'refund' in hits[0].lower() or 'return' in hits[0].lower(), 'the top chunk should be about refunds'
# the returned chunks are in descending-similarity order
qv = embed('how do I get a refund')
hit_scores = [cosine(qv, embed(c)) for c in hits]
assert hit_scores == sorted(hit_scores, reverse=True), 'chunks are ranked by similarity'
print('\n✅ retrieval grounds the answer in the relevant chunk')

Increase `k` and watch off-topic chunks (shipping, passwords) leak into the context — the precision/coverage trade-off from the lesson.

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **retrieval misses** | anything the retriever doesn't surface, the generator can't use |
| **k too large** | off-topic chunks dilute the prompt and cost tokens (demo) |
| **chunk size** | too big dilutes the signal, too small fragments the answer |
| **stale index** | the knowledge base drifts from the vector store; re-embed on updates |
| **no citation** | ungrounded answers are unverifiable; return the source chunks |

Demo: retrieval relevance falls off past the top chunk, so keep k small.

In [ ]:
# The k tradeoff, and why RAG beats a raw LLM here: a bigger k adds off-topic chunks
# (shipping, passwords) that dilute the prompt, while retrieval keeps the answer grounded
# in the source instead of the model's memory. We show relevance falls off past the top chunk.
qv2 = embed('how do I get a refund')
for k in [1, 2, 3, 4]:
    hits = retrieve('how do I get a refund', k=k)
    scores = [round(cosine(qv2, embed(c)), 3) for c in hits]
    print(f'k={k}: retrieved chunk scores = {scores}')
print('\nScores drop off after the top chunk -> a small k keeps the prompt on-topic;')
print('too large a k floods the context with irrelevant text (and burns tokens).')

## ✏️ Your turn

Implement `recall_at_k`: given the indices of the truly relevant chunks and the retrieved indices, return the fraction of relevant chunks that were retrieved.

In [ ]:
def recall_at_k(relevant, retrieved):
    # TODO(you): fraction of `relevant` indices that appear in `retrieved`.
    return 0.0

assert recall_at_k({0, 1}, [0, 3]) == 0.5
assert recall_at_k({2}, [2, 0, 1]) == 1.0
print('passed ✓')

<details><summary>Solution</summary>

```python
def recall_at_k(relevant, retrieved):
    relevant = set(relevant)
    hits = len(relevant & set(retrieved))
    return hits / len(relevant)
```

</details>

## Key takeaways

- **RAG = chunk → retrieve → generate:** the model answers from retrieved facts, not
  parametric memory — grounding that reduces hallucination.
- **Retrieval must surface the relevant chunk** (verified) — everything downstream
  depends on recall.
- **k trades grounding for noise:** too small misses the answer, too large floods the
  prompt with off-topic chunks and tokens (demo).
- **Chunking and embedding quality decide RAG quality** — the retrieval half is where
  most RAG systems win or lose.